This notebook builds a unified metadata configuration by combining Oracle information schema, key constraints, and primary key data, and generates structured source queries for downstream in Fabric.

In [5]:
ConfigSchemaName =''

StatementMeta(, 48e3567d-0035-430b-b9c8-25fb7f7389c7, 7, Finished, Available, Finished, False)

In [6]:
################################################### IMPORT  REQUIRED PACKAGES ###################################################
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
col, lit, concat, when, trim, collect_list, row_number, to_date,date_format,
col, concat, lit, to_timestamp, date_format, to_date, coalesce,concat_ws, md5,struct,
udf , concat, to_timestamp,count, when ,  sum as _sum)
from pyspark.sql.window import Window
from datetime import datetime, timezone, timedelta
from delta.tables import DeltaTable
import pandas as pd
from com.microsoft.spark.fabric import Constants
import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants
from pyspark.sql.types import (
    DoubleType, IntegerType, BooleanType, StringType,StructField,ArrayType,
    BinaryType, TimestampType, DateType, LongType, StructType
)
import concurrent.futures
import pytz
import traceback
import os
import json
 
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, concat, collect_list, row_number
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)
from pyspark.sql.window import Window

StatementMeta(, 48e3567d-0035-430b-b9c8-25fb7f7389c7, 8, Finished, Available, Finished, False)

In [9]:
################################################### SPARK SESSION ###################################################
spark = SparkSession.builder \
    .appName("Oracle") \
    .config("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED") \
    .config("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED") \
    .config("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED") \
    .config("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED") \
    .config("spark.sql.legacy.parquet.datetimeRebaseModeInWrite", "CORRECTED") \
    .config("spark.sql.legacy.parquet.datetimeRebaseModeInRead", "CORRECTED") \
    .config("spark.sql.caseSensitive", "true") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

StatementMeta(, 48e3567d-0035-430b-b9c8-25fb7f7389c7, 11, Finished, Available, Finished, False)

In [10]:
################################################### LOADING METADATA ####################################################
try:
    InformationSchema=f"SELECT * FROM {ConfigSchemaName}.SourceInformationSchema"
    InformationSchema_df= spark.read.option(Constants.DatabaseName, "WH_MetaData").synapsesql(InformationSchema)
   #InformationSchema_df=InformationSchema_df.filter(col("OWNER")=='HOSPITAL')
except Exception as e:
    print(e) 

StatementMeta(, 48e3567d-0035-430b-b9c8-25fb7f7389c7, 12, Finished, Available, Finished, False)

In [11]:
################################################### LOADING KEY CONSTRAINTS ################################################### 
try:
    KeyConstraint = f"SELECT * FROM {ConfigSchemaName}.KeyConstraintConfig"

    KeyConstraint_df = spark.read.option(
        Constants.DatabaseName,
        "WH_MetaData"
    ).synapsesql(KeyConstraint)

    #KeyConstraint_df = KeyConstraint_df.filter(col("OWNER") == 'HOSPITAL')

except Exception as e:
    print("Error loading Key Constraints:", e)

StatementMeta(, 48e3567d-0035-430b-b9c8-25fb7f7389c7, 13, Finished, Available, Finished, False)

In [12]:
################################################### LOADING PRIMARY KEY  ################################################### 

try:
    PrimaryKey = f"SELECT * FROM {ConfigSchemaName}.PrimayKeyConfig"

    PrimaryKey_df = spark.read.option(
        Constants.DatabaseName,
        "WH_MetaData"
    ).synapsesql(PrimaryKey)

    #PrimaryKey_df = PrimaryKey_df.filter(
       # col("OWNER") == 'HOSPITAL'
    #)

except Exception as e:
    print("Error loading Key Constraints:", e)

StatementMeta(, 48e3567d-0035-430b-b9c8-25fb7f7389c7, 14, Finished, Available, Finished, False)

In [13]:
 ################################################### 4.FILTER AND GROUP  ###################################################
information_req = InformationSchema_df
#.filter(col("OWNER") == "HOSPITAL"
#)

grouped_df = InformationSchema_df.groupBy("OWNER", "TABLE_NAME").agg(
    collect_list(struct("COLUMN_NAME", "DATA_TYPE")).alias("columns")
)

StatementMeta(, 48e3567d-0035-430b-b9c8-25fb7f7389c7, 15, Finished, Available, Finished, False)

In [14]:
from pyspark.sql.functions import collect_set

pk_df = PrimaryKey_df.groupBy("OWNER", "TABLE_NAME").agg(
    collect_set("COLUMN_NAME").alias("PrimaryKey")
)

final_with_pk = grouped_df.join(
    pk_df,
    ["OWNER", "TABLE_NAME"],
    "left"
)

StatementMeta(, 48e3567d-0035-430b-b9c8-25fb7f7389c7, 16, Finished, Available, Finished, False)

In [15]:
################################################### 5.GENERATING SOURCE QUERY ###################################################
def generate_source_query(table_name, columns, schema_name):
    select_statements = []

    for column in columns:
        column_name = column["COLUMN_NAME"]
        data_type = column["DATA_TYPE"]


        if data_type == "NUMBER":
            select_statements.append(f"CAST({column_name} AS BINARY_DOUBLE) AS {column_name}")
        elif data_type == "DATE":
            select_statements.append(f"CAST({column_name} AS TIMESTAMP) AS {column_name}")
        else:
            select_statements.append(column_name)

    return f"SELECT {', '.join(select_statements)} FROM {schema_name}.{table_name}"

StatementMeta(, 48e3567d-0035-430b-b9c8-25fb7f7389c7, 17, Finished, Available, Finished, False)

In [16]:
################################################### 6.COLLECTING METADATA ###################################################
metadata = []

file_datatypes = [
    "CLOB",
    "NCLOB",
    "BLOB",
    "RAW",
    "ROWID",
    "UROWID",
    "BFILE"
]


for row in final_with_pk.collect():
    table_name = row["TABLE_NAME"]
    schema_name = row["OWNER"]
    columns = row["columns"]

    source_query = generate_source_query(table_name, columns, schema_name)
    primary_key = ",".join(row["PrimaryKey"]) if row["PrimaryKey"] else None

    
    Is_File = any(
        col["DATA_TYPE"] in file_datatypes
        for col in columns
    )

    
    load_type = "File" if Is_File else "Table"
    metadata.append((
        table_name,
        schema_name,
        source_query,
        primary_key,
        load_type
    ))

StatementMeta(, 48e3567d-0035-430b-b9c8-25fb7f7389c7, 18, Finished, Available, Finished, False)

In [17]:
schema = StructType([
    StructField("SourceTableName", StringType(), True),
    StructField("SourceSchemaName", StringType(), True),
    StructField("SourceQuery", StringType(), True),
    StructField("PrimaryKey", StringType(), True),
    StructField("LoadType", StringType(), True)
])

df = spark.createDataFrame(metadata, schema=schema)

StatementMeta(, 48e3567d-0035-430b-b9c8-25fb7f7389c7, 19, Finished, Available, Finished, False)

In [18]:

try:
    Configdf="SELECT Id  FROM Oracle.OneTimeConfigETL "
    max_id= spark.read.option(Constants.DatabaseName, "WH_MetaData").synapsesql(Configdf)
    max_id = max_id.agg({"Id": "max"}).collect()[0]["max(Id)"]
except Exception as e:
    max_id=0

StatementMeta(, 48e3567d-0035-430b-b9c8-25fb7f7389c7, 20, Finished, Available, Finished, False)

In [19]:

################################################## 8.COLUMN SELECTION AND MODIFICATIONS ###################################################
 
window_spec = Window.orderBy(lit(1))
 
from pyspark.sql.functions import lit, concat, row_number, col
from pyspark.sql.window import Window

 
# Update the DataFrame with the new Id starting from 294
df = df.withColumn("Id", row_number().over(window_spec) + max_id) \
    .withColumn("BronzeSchemaName", concat(col("SourceSchemaName"))) \
    .withColumn("BronzeTableName", col("SourceTableName")) \
    .withColumn("SilverSchemaName", concat(col("SourceSchemaName"))) \
    .withColumn("SilverTableName", col("SourceTableName")) \
    .withColumn("IsActive", lit("1")) 
#final_df = df.withColumn("Id", col("Id").cast(IntegerType()))
 
selected_df = df.select(
    "Id", "SourceTableName", "SourceSchemaName", "BronzeSchemaName",
    "BronzeTableName", "SilverSchemaName", "SilverTableName",
    "SourceQuery", "IsActive","LoadType","PrimaryKey"
)
 
selected_df = selected_df.withColumn(
    "CreatedBy",lit("ETL Developer")
)
createdtime = datetime.now(timezone.utc)
selected_df = selected_df.withColumn(
    "CreatedDate",lit(createdtime).cast(StringType())
)

StatementMeta(, 48e3567d-0035-430b-b9c8-25fb7f7389c7, 21, Finished, Available, Finished, False)

In [20]:
################################################### 9. DEFINING STRUCTURE OF THE DATAFRAME ###################################################
 
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
new_schema = StructType([
    StructField("Id", IntegerType(), True),
    StructField("SourceSchemaName", StringType(), True),
    StructField("SourceTableName", StringType(), True),
    StructField("BronzeSchemaName", StringType(), True),
    StructField("BronzeTableName", StringType(), True),
    StructField("SilverSchemaName", StringType(), True),
    StructField("SilverTableName", StringType(), True),
    StructField("SourceQuery", StringType(), True),
    StructField("LoadType", StringType(), True), 
    StructField("PrimaryKey", StringType(), True),
    StructField("CreatedBy", StringType(), True),
    StructField("CreatedDate", StringType(), True),
    StructField("IsActive", StringType(), True),
])

df = selected_df.withColumn("Id", col("Id").cast(IntegerType())) \
                .withColumn("CreatedBy", lit("Oracle_Config_Notebook")) \
                .withColumn("CreatedDate", lit(datetime.now(timezone.utc)).cast(StringType())) \
                .select(
                    "Id",
                    "SourceSchemaName",
                    "SourceTableName",
                    "BronzeSchemaName",
                    "BronzeTableName",
                    "SilverSchemaName",
                    "SilverTableName",
                    "SourceQuery",
                    "LoadType",
                    "PrimaryKey",
                    "CreatedBy",
                    "CreatedDate",
                    "IsActive",
                )

df = spark.createDataFrame(df.rdd, new_schema)

StatementMeta(, 48e3567d-0035-430b-b9c8-25fb7f7389c7, 22, Finished, Available, Finished, False)

In [21]:
display(df)

StatementMeta(, 48e3567d-0035-430b-b9c8-25fb7f7389c7, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d09b458e-476f-4cc2-bfd2-97bccd70d069)

In [22]:
################################################### 9.LOADING THE METADATA DETAILS ###################################################
df.write \
    .mode("append") \
    .synapsesql(f"WH_MetaData.{ConfigSchemaName}.OneTimeConfigETL")

StatementMeta(, 48e3567d-0035-430b-b9c8-25fb7f7389c7, 24, Finished, Available, Finished, False)